In [1]:
%reload_ext autoreload
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
import IPython.display as ipd
import whisper
import sys
sys.path.append("/home/romolo/VT1/coqui-tts")
from model_conf import ModelPaths, load_tts_and_trainer

/home/romolo/VT1/coqui-tts/.venv/lib/python3.11/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/home/romolo/VT1/coqui-tts/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/romolo/VT1/coqui-tts/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


In [3]:
paths = ModelPaths()
tts, model, train_model, config = load_tts_and_trainer(paths)

 > Using model: xtts
>> DVAE weights restored from: /home/romolo/VT1/coqui-tts/XTTS_v2.0_original_model_files/dvae.pth


In [4]:
asr_model = whisper.load_model("base")

In [5]:
import os

In [6]:
from TTS.tts.models.xtts import load_audio
import soundfile as sf

In [7]:
from huggingface_hub import hf_hub_download

# automatically checks for cached file, optionally set `cache_dir` location
model_file = hf_hub_download(repo_id='Jenthe/ECAPA2', filename='ecapa2.pt', cache_dir=None)

In [8]:
import torch
import torchaudio
import torch.nn.functional as F

ecapa2 = torch.jit.load(model_file, map_location='cuda')


In [9]:
paths = ModelPaths()
tts, model, train_model, config = load_tts_and_trainer(paths)

 > Using model: xtts
>> DVAE weights restored from: /home/romolo/VT1/coqui-tts/XTTS_v2.0_original_model_files/dvae.pth


In [10]:
import pandas as pd

In [11]:
df = pd.read_csv('/data/dev/metadata/0000.csv')
df.head()

,id,orig_path,start,end,text,speaker_id,book_id,segment_id,dataset,split,store_id,estimated_size_bytes
0,10362_11341_000000,http://www.archive.org/download/glinni_sacri_1...,195.34,206.13,tanto d'ogni laudato esser la prima di dio la ...,10362,11341,0,mls_italian,dev,mls_italian_dev_10362_11341_000000,951678.0
1,10362_11341_000001,http://www.archive.org/download/glinni_sacri_1...,42.59,54.46,qual angolo ti raccogliea nascente quando il t...,10362,11341,1,mls_italian,dev,mls_italian_dev_10362_11341_000001,1046934.0
2,10362_11341_000002,http://www.archive.org/download/glinni_sacri_1...,164.34,179.28,un estranio giovinetto si posò sul monumento e...,10362,11341,2,mls_italian,dev,mls_italian_dev_10362_11341_000002,1317708.0
3,10362_11341_000003,http://www.archive.org/download/glinni_sacri_1...,183.58,196.81,come vittima innanzi all'altar non lo seppe il...,10362,11341,3,mls_italian,dev,mls_italian_dev_10362_11341_000003,1166886.0
4,10362_11341_000004,http://www.archive.org/download/glinni_sacri_1...,179.31,195.34,anco ogni giorno se ne parla e tanto secol vi ...,10362,11341,4,mls_italian,dev,mls_italian_dev_10362_11341_000004,1413846.0


In [12]:
hdf5path = "/data/dev/audios/0000.hdf5"

In [13]:
df.store_id.iloc[11000]

'mls_english_dev_10326_10194_000016'

In [14]:
df.store_id.iloc[11100]

'mls_english_dev_8862_10240_000002'

In [15]:
from TTS.tts.models.xtts import load_audio

In [16]:
target = load_audio(df.store_id.iloc[11000],22050,hdf5path)
ipd.Audio(target,rate=22050)

In [17]:
refrence = load_audio(df.store_id.iloc[11100],22050,hdf5path)
ipd.Audio(refrence,rate=22050)

In [36]:
output1 = model.forward_iteration_hdf5('en',df.text.iloc[11000],df.store_id.iloc[11000],df.store_id.iloc[11100],train_model,config.model_args.max_conditioning_length, config.model_args.min_conditioning_length,tts,ecapa2,asr_model,target_hdf5_path=hdf5path,ref_hdf5_path=hdf5path)

Cleaned up temp HDF5 file: /tmp/iter_temp_75440_1764237318.hdf5


In [39]:
ipd.Audio(output1[3][0],rate=24000)

In [20]:

orig_target_sample = '/home/romolo/VT1/coqui-tts/test_data/Dataset/references_new/EN_1624/1624-142933-0000.wav'
ref_samples = ['/home/romolo/VT1/coqui-tts/test_data/Dataset/references_new/EN_1447/1447-17506-0000.wav']

In [21]:
text = asr_model.transcribe(orig_target_sample)["text"]

In [22]:
output = model.forward_iteration('en',text,orig_target_sample,ref_samples,train_model,config.model_args.max_conditioning_length, config.model_args.min_conditioning_length,tts,ecapa2,asr_model)

Iteration 0: BLUE: 1.0, WER: 0.0, Target Cosine: tensor([-0.0086], device='cuda:0'), Reference Cosine: tensor([0.1886], device='cuda:0')
Iteration 1: BLUE: 0.10073051057692405, WER: 0.6666666666666666, Target Cosine: tensor([0.0103], device='cuda:0'), Reference Cosine: tensor([0.3318], device='cuda:0')
Iteration 2: BLUE: 1.0, WER: 0.0, Target Cosine: tensor([-0.0657], device='cuda:0'), Reference Cosine: tensor([0.3776], device='cuda:0')
Iteration 3: BLUE: 1.0, WER: 0.0, Target Cosine: tensor([-0.0431], device='cuda:0'), Reference Cosine: tensor([0.3550], device='cuda:0')
Iteration 4: BLUE: 0.10073051057692405, WER: 0.6666666666666666, Target Cosine: tensor([-0.0818], device='cuda:0'), Reference Cosine: tensor([0.3369], device='cuda:0')


In [23]:
ipd.Audio(output[3][2],rate=20400)

In [24]:
from run_test import segment_quality_score

In [25]:
tar_audio, sr = torchaudio.load(orig_target_sample) # sample rate of 16 kHz expected
tar_audio = torchaudio.functional.resample(tar_audio, orig_freq=sr, new_freq=16_000)
tar_embedding = ecapa2(tar_audio.to('cuda'))

ref_audio, sr2 = torchaudio.load(ref_samples[0]) # sample rate of 16 kHz expected
ref_audio = torchaudio.functional.resample(ref_audio, orig_freq=sr2, new_freq=16_000)
ref_embedding = ecapa2(ref_audio.to('cuda'))

In [26]:
scores, info = segment_quality_score(asr_model, tar_embedding, ref_embedding,output[3][0],ecapa2)

In [27]:
info

{0: {'overall_quality': 0.519412424787879,
  'wer': 0.0,
  'blue': 1.0,
  'target_sim': 0.010083328932523727,
  'ref_sim': 0.20406237244606018},
 1: {'overall_quality': 0.4250415533781051,
  'wer': 1.0,
  'blue': 0,
  'target_sim': -0.08810341358184814,
  'ref_sim': 0.16435088217258453}}

In [28]:
def calc_sim(aud1,aud2):
    #cossim between embeddings
    audio, sr = torchaudio.load(aud1)
    audio = torchaudio.functional.resample(audio, orig_freq=sr, new_freq=16_000)# sample rate of 16 kHz expected
    embedding = ecapa2(audio.to('cuda'))
    ref_audio, sr = torchaudio.load(aud2) # sample rate of 16 kHz expected
    ref_audio = torchaudio.functional.resample(ref_audio, orig_freq=sr, new_freq=16_000)
    ref_embedding = ecapa2(ref_audio.to('cuda'))
    sim = F.cosine_similarity(embedding, ref_embedding)
    return sim

In [29]:
for i in range(0,10):
    targ = calc_sim(f"/home/romolo/VT1/coqui-tts/data/outputs/iterate_output_{i}.wav","/home/romolo/VT1/coqui-tts/data/target.wav")
    print(f"sample {i}")
    print(f"sim -- target to output: {targ}")
    ref = calc_sim(f"/home/romolo/VT1/coqui-tts/data/outputs/iterate_output_{i}.wav",ref_samples[0])
    print(f"sim -- ref to output: {ref}")


RuntimeError: Failed to open the input "/home/romolo/VT1/coqui-tts/data/outputs/iterate_output_0.wav" (No such file or directory).
Exception raised from get_input_format_context at /__w/audio/audio/pytorch/audio/src/libtorio/ffmpeg/stream_reader/stream_reader.cpp:42 (most recent call first):
frame #0: c10::Error::Error(c10::SourceLocation, std::string) + 0x96 (0x7f0b50b6c1b6 in /home/romolo/VT1/coqui-tts/.venv/lib/python3.11/site-packages/torch/lib/libc10.so)
frame #1: c10::detail::torchCheckFail(char const*, char const*, unsigned int, std::string const&) + 0x64 (0x7f0b50b15a76 in /home/romolo/VT1/coqui-tts/.venv/lib/python3.11/site-packages/torch/lib/libc10.so)
frame #2: <unknown function> + 0x42034 (0x7f09dca19034 in /home/romolo/VT1/coqui-tts/.venv/lib/python3.11/site-packages/torio/lib/libtorio_ffmpeg4.so)
frame #3: torio::io::StreamingMediaDecoder::StreamingMediaDecoder(std::string const&, std::optional<std::string> const&, std::optional<std::map<std::string, std::string, std::les